# Function Tools

The Leaf Village Assistant has been answering from the model's own (possibly
made-up) knowledge. This notebook gives it real tools: `get_ninja_rank`,
`check_mission_status`, and `scout_mission`.

Gotcha up front: `agents.tool` is _not_ the `@tool` decorator alias — it's the
`agents/tool.py` submodule, which shadows the decorator at the top-level `agents`
namespace. This notebook imports the unambiguous name instead: `from agents import
function_tool`.


## `get_ninja_rank` — a tool that reads run context

If a tool's first parameter is a `RunContextWrapper`, the SDK excludes it from the
tool's JSON schema (the model never sees it) and passes through whatever
`context=` the run was given. The docstring below isn't just documentation — the
SDK parses it to build the tool's description and the `ninja_name` field's
description in the JSON schema.


In [ ]:
from dataclasses import (
    dataclass,  # a decorator that makes a simple class that holds data
)

from agents import (
    RunContextWrapper,  # holds the context= value you passed, and gives it to callbacks
    function_tool,  # turns a normal function into a tool an Agent can call
)


# context= data: passed to Runner.run(context=NinjaContext(...)) below, so any
# tool whose first parameter is a RunContextWrapper[NinjaContext] can read it
@dataclass
class NinjaContext:
    caller_name: str  # who is asking. It is passed as context= on Runner.run and is not part of what they ask


# Define the tool
# @function_tool callback: looks up a ninja's rank, and shows that a
# RunContextWrapper first parameter is left out of the tool's JSON schema on its own
@function_tool
def get_ninja_rank(ctx: RunContextWrapper[NinjaContext], ninja_name: str) -> str:
    """Look up a ninja's rank in the Village registry.

    Args:
        ninja_name: The ninja's full name, e.g. "Naruto Uzumaki".
    """

    _NINJA_RANKS: dict[str, str] = {
        "Naruto Uzumaki": "Hokage",
        "Sasuke Uchiha": "Shadow Hokage",
        "Tsunade": "Hokage",
    }
    rank = _NINJA_RANKS.get(ninja_name)
    if rank is None:
        return f"No ninja named '{ninja_name}' is registered with the Village."
    return f"{ctx.context.caller_name} looked up {ninja_name}: rank {rank}."


print("[schema]", get_ninja_rank.params_json_schema)

## `check_mission_status` — Pydantic `Field` constraints

`Annotated[int, Field(ge=1, le=10)]` shows up directly in the tool's JSON schema
as `minimum`/`maximum` — the model provider enforces the range, not your Python
code.


In [ ]:
from typing import (
    Annotated,  # adds extra info (like a Field) to one parameter
)

from agents import (
    function_tool,  # turns a normal function into a tool an Agent can call
)
from pydantic import (
    Field,  # adds rules to one parameter, like min, max, and a description
)


# Define the tool
# @function_tool callback: checks how dangerous a mission is, and shows that
# Annotated[T, Field(...)] rules appear right in the JSON schema
@function_tool
def check_mission_status(
    mission_rank: Annotated[str, Field(description="Mission rank letter, Genin through Kage")],
    danger_level: Annotated[int, Field(ge=1, le=10, description="1 (safe) to 10 (extreme)")],
) -> str:
    """Check a mission's current status before dispatching ninjas."""
    # 8 or higher means a full evacuation call, not the normal clear procedure
    if danger_level >= 8:
        return f"Rank-{mission_rank} mission at danger {danger_level}: EVACUATE, Kage-level team required."
    return f"Rank-{mission_rank} mission at danger {danger_level}: standard clear procedure."


print("[danger_level schema]", check_mission_status.params_json_schema["properties"]["danger_level"])

## `scout_mission` — async-only timeouts and custom failure handling

Two real gotchas: `timeout=` only works on an `async def` tool (a sync one raises
`ValueError` at decoration time), and the default failure behavior turns an
unhandled exception into a message _for the model_ instead of crashing the run —
`failure_error_function=` customizes that message.


In [ ]:
import asyncio  # makes scout_mission pause, like a real scouting trip

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunContextWrapper,  # holds the context= value you passed, and gives it to callbacks
    function_tool,  # turns a normal function into a tool an Agent can call
)


# failure_error_function callback: passed to @function_tool(failure_error_function=...)
# below. It runs when the tool fails, and replaces the SDK's default error
# message with a Naruto-style one the model can read
def _mission_scan_failed(
    ctx: RunContextWrapper,  # the run's context. It is not used here, but it is always passed
    error: Exception,  # the error the tool raised
) -> str:
    return f"[System] Mission scan failed: {error}. Try scouting again."


# Define the tool
# @function_tool callback: an async tool with a real timeout= and a custom
# failure_error_function=, which shows both gotchas of this episode
@function_tool(timeout=2.0, failure_error_function=_mission_scan_failed)
async def scout_mission(mission_id: str) -> str:
    """Send a scout ninja into a mission zone and report back (can time out or fail)."""
    await asyncio.sleep(0.1)
    if mission_id == "Mission-13":
        raise RuntimeError("unstable chakra reading — scan aborted")
    return f"{mission_id} scouted: no immediate threats detected."


# Define the tool
# @function_tool callback: the same as scout_mission, but its timeout is
# shorter than its own pause on purpose, so the demo below times out
@function_tool(timeout=0.01)  # much less than the 0.1 second pause above, so it times out when we want
async def scout_mission_slow(mission_id: str) -> str:
    """Same as scout_mission, but with a timeout too short to ever succeed."""
    await asyncio.sleep(0.1)
    return f"{mission_id} scouted."


# Initialize the agent
ninja_tools_agent = Agent(
    name="Leaf Village Assistant",
    instructions="Use your tools to look up ninja ranks, check mission status, and scout missions.",
    # tools: the list of function_tool functions this agent can call
    tools=[
        get_ninja_rank,
        check_mission_status,
        scout_mission,
        scout_mission_slow,
    ],
)
print("[tools]", [t.name for t in ninja_tools_agent.tools])

## Run it, and prove both gotchas — offline

Three scripted runs through the same agent: a normal tool call, `scout_mission_slow`
timing out, and `scout_mission` failing on `Mission-13` — proving the whole loop and
both gotchas with no live API call.


In [ ]:
from agents import (
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
    function_call,  # makes one scripted tool-call step for ScriptedModel
)


# Demo: three scripted runs that prove a normal tool call, a timeout, and a custom tool error message, all offline
async def demo_tool_calls() -> None:
    # Run the Agent — a normal tool call
    result = await Runner.run(
        ninja_tools_agent,
        "What rank is Naruto Uzumaki?",
        # context: your own value that tools and callbacks can read during the run
        context=NinjaContext(caller_name="Shikamaru Nara"),
        # run_config: settings for just this run
        run_config=RunConfig(
            # model: use this model for this run only
            model=ScriptedModel(
                [
                    [function_call("get_ninja_rank", {"ninja_name": "Naruto Uzumaki"}, call_id="c1")],
                    [assistant_message("Naruto Uzumaki is Kage-level.")],
                ]
            ),
            # workflow_name: the name this run shows up under in tracing
            workflow_name="e03-tool-call-demo",
        ),
    )
    # proves the tool really runs (context= reaches
    # get_ninja_rank's ctx.context.caller_name) before the agent answers
    print("[final]", result.final_output)

    # Run the Agent — scout_mission_slow times out
    timeout_result = await Runner.run(
        ninja_tools_agent,
        "scout Mission-01 quickly",
        # run_config: settings for just this run
        run_config=RunConfig(
            # model: use this model for this run only
            model=ScriptedModel(
                [
                    [function_call("scout_mission_slow", {"mission_id": "Mission-01"}, call_id="c2")],
                    [assistant_message("handled the timeout")],
                ]
            ),
            # workflow_name: the name this run shows up under in tracing
            workflow_name="e03-timeout-demo",
        ),
    )
    # proves timeout= turns a slow async tool into a clean tool-output message
    # for the model to read, instead of freezing the whole run
    for item in timeout_result.new_items:
        if item.type == "tool_call_output_item":
            print("[timeout]", item.output)

    # Run the Agent — scout_mission fails on Mission-13
    failure_result = await Runner.run(
        ninja_tools_agent,
        "scout Mission-13",
        # run_config: settings for just this run
        run_config=RunConfig(
            # model: use this model for this run only
            model=ScriptedModel(
                [
                    [function_call("scout_mission", {"mission_id": "Mission-13"}, call_id="c3")],
                    [assistant_message("handled the failure")],
                ]
            ),
            # workflow_name: the name this run shows up under in tracing
            workflow_name="e03-failure-demo",
        ),
    )
    # proves failure_error_function= replaces the SDK's default error text with
    # our own _mission_scan_failed message
    for item in failure_result.new_items:
        if item.type == "tool_call_output_item":
            print("[failure]", item.output)


await demo_tool_calls()
# Outside a notebook: asyncio.run(demo_tool_calls())

## Run it live — optional, needs a real key


In [ ]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# Demo: the same rank lookup and mission check with the real API, using all four tools together
async def demo_live() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Run the Agent
    live_result = await Runner.run(
        ninja_tools_agent,
        "What rank is Naruto Uzumaki, and is Mission-07 safe to scout at danger level 9?",
        # context: your own value that tools and callbacks can read during the run
        context=NinjaContext(caller_name="Shikamaru Nara"),
    )

    for item in live_result.new_items:
        if item.type == "tool_call_output_item":
            print("Tool call response:", item.output)
    print("[live]", live_result.final_output)


await demo_live()
# Outside a notebook: asyncio.run(demo_live())

## Summary

- `from agents import function_tool` is the unambiguous way to get the decorator —
  `agents.tool` is a submodule, not the alias — and a `RunContextWrapper` first
  parameter is auto-excluded from the tool's schema while still receiving the
  real context.
- `Annotated[T, Field(...)]` constraints show up directly in the generated JSON
  schema; `timeout=` requires an `async def` tool, and `failure_error_function=`
  customizes what the model sees when the tool raises.
